In [ ]:
#parsing HTML page 1 zameen.com

from bs4 import BeautifulSoup

file_path = "C:/Users/Ouss/Desktop/My Projects/P 1/Flats for Rent in Karachi - Zameen.com Page 1.htm"

with open(file_path, "r", encoding="utf-8") as f:
    soup = BeautifulSoup(f.read(),"lxml")


articles = soup.find_all("article")
list_items = soup.find_all("li", role="article")

print(f"Total <article> tags found: {len(articles)}")
print(f"Total <li role='article'> tags found: {len(list_items)}")




In [ ]:

first_card = list_items[0]

if first_card.find('h2'):
    print(f"H2 Tag Text: {first_card.find('h2').text.strip()}")
elif first_card.find('h3'):
    print(f"H3 Tag Text: {first_card.find('h3').text.strip()}")

print("\n--- ALL TEXT INSIDE THIS CARD ---")

print(first_card.get_text(separator=" | ").strip())



H2 Tag Text: Brand New 2 Bed DD | Single Balcony | Sawera Enclave Available For Rent

--- ALL TEXT INSIDE THIS CARD ---
super hot | 12 | Share on Facebook | Share on Twitter | Share on WhatsApp | Send via GMail | Send via E-Mail | Titanium | PKR | 60 Thousand | Sawera Enclave, Suparco Road | 2 | 2 | 1,100 sqft | Brand New 2 Bed DD | Single Balcony | Sawera Enclave Available For Rent | Brand New 2 Bed DD with Single Balc | ... | more | Added: 4 days ago | (Updated: 2 hours ago) | WhatsApp | Call | 
 | 
 |


In [ ]:
import pandas as pd
import re

properties_data = []

def clean_pkr_price(price_str):
    """Parses text strings like '60 Thousand' or '1.5 Lakh' into numerical integers."""
    price_str = price_str.lower().strip()
    # Extract numerical values (including decimals)
    match = re.search(r"[-+]?\d*\.\d+|\d+", price_str)
    if not match:
        return 0
    val = float(match.group())
    
    if "thousand" in price_str:
        return int(val * 1000)
    elif "lakh" in price_str:
        return int(val * 100000)
    elif "crore" in price_str:
        return int(val * 10000000)
    return int(val)

def clean_area(area_str):
    """Removes commas and 'sqft' characters to return a clean integer."""
    # Remove commas and extract numbers
    clean_str = area_str.replace(",", "")
    match = re.search(r"\d+", clean_str)
    return int(match.group()) if match else 0

# Loop through all 25 isolated list elements
for idx, card in enumerate(list_items):
    try:
        # 1. Extract Title
        title = card.find('h2').text.strip() if card.find('h2') else "N/A"
        
        # Break text into a clean list of segments for targeted searching
        segments = [seg.strip() for seg in card.get_text(separator="|").split("|") if seg.strip()]
        
        price_raw = "N/A"
        location = "N/A"
        area_raw = "N/A"
        
        # 2. Extract Price and Location based on 'PKR' anchor position
        if "PKR" in segments:
            pkr_index = segments.index("PKR")
            if pkr_index + 1 < len(segments):
                price_raw = segments[pkr_index + 1]
            if pkr_index + 2 < len(segments):
                location = segments[pkr_index + 2]
                
        # 3. Extract Area by searching for the segment containing 'sqft'
        for seg in segments:
            if "sqft" in seg.lower():
                area_raw = seg
                break
        
        # 4. Apply Cleaning Functions
        price_pkr = clean_pkr_price(price_raw)
        area_sqft = clean_area(area_raw)
        
        # 5. Append structured dictionary to master list
        properties_data.append({
            "Listing_ID": 1000 + idx, # Generates a baseline tracking ID
            "Title": title,
            "Price_Raw": price_raw,
            "Price_PKR": price_pkr,
            "Area_SqFt": area_sqft,
            "Location": location
        })
        
    except Exception as e:
        print(f"Skipping row {idx} due to unexpected structure anomaly: {e}")
        continue

df = pd.DataFrame(properties_data)

# Print execution summary and top rows
print(f"Successfully processed {len(df)} rows into the DataFrame.")
print("\n--- DataFrame Sample Preview ---")
print(df[["Listing_ID", "Price_PKR", "Area_SqFt", "Location"]].head())

# Export to CSV
df.to_csv("clean_karachi_apartments.csv", index=False)

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
import time
import random

COOKIE_STRING = "YOUR_COPIED_COOKIE_STRING_HERE" 

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.5",
    "Connection": "keep-alive",
    "Cookie": COOKIE_STRING
}

def clean_pkr_price(price_str):
    price_str = price_str.lower().strip()
    match = re.search(r"[-+]?\d*\.\d+|\d+", price_str)
    if not match:
        return 0
    val = float(match.group())
    if "thousand" in price_str:
        return int(val * 1000)
    elif "lakh" in price_str:
        return int(val * 100000)
    elif "crore" in price_str:
        return int(val * 10000000)
    return int(val)

def clean_area(area_str):
    clean_str = area_str.replace(",", "")
    match = re.search(r"\d+", clean_str)
    return int(match.group()) if match else 0

# Master array to store all 1,190 records
all_properties = []
listing_counter = 1000

START_PAGE = 1
END_PAGE = 47

print("🚀 Initializing Live Multi-Page Crawler...")

for page_num in range(START_PAGE, END_PAGE + 1):
    # Zameen pagination URL schema: page number appended right before the extension
    url = f"https://www.zameen.com/Rentals_Flats_Apartments/2bed-rent/Karachi-2-{page_num}.html"
    
    print(f"Fetching Page {page_num}/{END_PAGE}...")
    
    try:
        response = requests.get(url, headers=HEADERS, timeout=15)
        
        if response.status_code == 403:
            print("❌ Cloudflare Blocked the request (403). Your session cookie has expired. Refresh your browser and update the COOKIE_STRING.")
            break
        elif response.status_code != 200:
            print(f"⚠️ Page {page_num} skipped. HTTP Status: {response.status_code}")
            continue
            
        soup = BeautifulSoup(response.content, "lxml")
        list_items = soup.find_all("li", role="article")
        
        page_listings_count = len(list_items)
        print(f"   Successfully parsed {page_listings_count} listings.")
        
        if page_listings_count == 0:
            print("   No listings found. Terminating loop early.")
            break
            
        for card in list_items:
            try:
                title = card.find('h2').text.strip() if card.find('h2') else "N/A"
                segments = [seg.strip() for seg in card.get_text(separator="|").split("|") if seg.strip()]
                
                price_raw = "N/A"
                location = "N/A"
                area_raw = "N/A"
                
                if "PKR" in segments:
                    pkr_index = segments.index("PKR")
                    if pkr_index + 1 < len(segments):
                        price_raw = segments[pkr_index + 1]
                    if pkr_index + 2 < len(segments):
                        location = segments[pkr_index + 2]
                        
                for seg in segments:
                    if "sqft" in seg.lower():
                        area_raw = seg
                        break
                
                price_pkr = clean_pkr_price(price_raw)
                area_sqft = clean_area(area_raw)
                
                all_properties.append({
                    "Listing_ID": listing_counter,
                    "Title": title,
                    "Price_Raw": price_raw,
                    "Price_PKR": price_pkr,
                    "Area_SqFt": area_sqft,
                    "Location": location
                })
                listing_counter += 1
                
            except Exception:
                continue
                
        # Defensive anti-scraping delay: pause the execution randomly between 3 to 7 seconds per page
        delay = random.uniform(3.0, 7.0)
        time.sleep(delay)
        
    except Exception as e:
        print(f"❌ Error scraping page {page_num}: {e}")
        break

# --- DATA CONSOLIDATION LAYER ---
master_df = pd.DataFrame(all_properties)

# Remove any logical errors where data wasn't captured
master_df = master_df.dropna(subset=["Price_PKR", "Area_SqFt"])
master_df = master_df[master_df["Price_PKR"] > 0]

print("\n📊 Extraction Phase Complete!")
print(f"Total Rows Compiled: {len(master_df)}")

# Complete dataset for the SQL stage
master_df.to_csv("clean_karachi_apartments.csv", index=False)